In [1]:
from samap.mapping import SAMAP
from samap.analysis import (get_mapping_scores, GenePairFinder, transfer_annotations,
                            sankey_plot, chord_plot, CellTypeTriangles, 
                            ParalogSubstitutions, FunctionalEnrichment,
                            convert_eggnog_to_homologs, GeneTriangles)
from samalg import SAM
import pandas as pd
from Bio import SeqIO
from samap.utils import (save_samap, load_samap)
import scanpy as sc
import pickle
from tqdm import tqdm

/scratch/miniconda/lib/python3.7/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [1]:
!python -c "import pandas; print(pandas.__version__)"

1.3.5


In [6]:
members = range(2,30)
for i in tqdm(members):
    sm = load_samap('CC/sm_Allen_Full_cc_cleaned_04162026_subclass_250_' + str(i) + '.pkl')
    org = 'cc'
    ref = 'mg'
    subclass_lc = pd.read_csv('CC/subclass_crossed_lc_cc_cleaned_12282025.csv', index_col = 'Unnamed: 0')
    sm.sams[org].adata.obs['subclass_lc'] = subclass_lc
    
    keys = {org:'subclass_lc',ref:'subclass_id_label'}
    D,MappingTable = get_mapping_scores(sm,keys)

    lim_MappingTable = MappingTable.filter(like=org + '_')
    lim_MappingTable = lim_MappingTable[lim_MappingTable.index.str.contains(ref)]
        
    mapping_dict = {}
    for item in lim_MappingTable:
        len_item = len(sm.sams[org].adata[sm.sams[org].adata.obs['subclass_lc'] == int(item[3:])])
        if len_item > 25:
            mapping_dict[item] = str(lim_MappingTable[item].idxmax())
        else:
            mapping_dict[item] = ref + '_Unlabeled'
            
    new_mapping = []
    for item in sm.sams[org].adata.obs['subclass_lc']:
        new_mapping.append(mapping_dict[org + '_' + str(item)][3:])
        
    df = pd.DataFrame(data = new_mapping, index = sm.sams[org].adata.obs_names, columns = [str(i)])
    df.to_csv('CC/CC_MG_mapping_cleaned_04192026_'+str(i)+'.csv')

100%|████████████████████████████████████████| 28/28 [1:16:18<00:00, 163.51s/it]


In [4]:
sm = load_samap('sm_Allen_NN_250_RV_cleaned_01142026.pkl')
org = 'rv'
ref = 'mg'
subclass_lc = pd.read_csv('subclass_crossed_lc_rv_cleaned_01142026.csv', index_col = 'Unnamed: 0')
sm.sams[org].adata.obs['subclass_lc'] = subclass_lc

keys = {org:'subclass_lc',ref:'subclass_id_label'}
D,MappingTable = get_mapping_scores(sm,keys)

lim_MappingTable = MappingTable.filter(like=org + '_')
lim_MappingTable = lim_MappingTable[lim_MappingTable.index.str.contains(ref)]

mapping_dict = {}
for item in lim_MappingTable:
    len_item = len(sm.sams[org].adata[sm.sams[org].adata.obs['subclass_lc'] == int(item[3:])])
    if len_item > 25:
        mapping_dict[item] = str(lim_MappingTable[item].idxmax())
    else:
        mapping_dict[item] = ref + '_Unlabeled'

new_mapping = []
for item in sm.sams[org].adata.obs['subclass_lc']:
    new_mapping.append(mapping_dict[org + '_' + str(item)][3:])

df = pd.DataFrame(data = new_mapping, index = sm.sams[org].adata.obs_names, columns = ['0'])
df.to_csv('RV_MG_mapping_cleaned_04092026_0.csv')